---
title: "Preprocessing Data"
subtitle: "Data import and joining"
description: "This notebook was created by a previous bachelor student who used for his/her thesis"
author: ["Unknown"]
categories: ["Python"]
date: 2025-03-03
---

# Data cleaning

Starting from the xlsx files, let's convert them to `.csv`:

### Imports

In [ ]:
import os
import pandas as pd

In [ ]:
# Setting directories containing data:
input_folder = "../../DATA/XLSX"
output_folder = "../../DATA/CSV"

# Creating output directory if it doesn't exist:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Iterating over each file in the input directory:
for filename in os.listdir(input_folder):
    if filename.endswith(".xlsx"):

        # Creating full path to the xlsx file:
        xlsx_path = os.path.join(input_folder, filename)

        # Reading the xlsx file:
        try:
            df = pd.read_excel(xlsx_path, engine="openpyxl")
        except Exception as e:
            print(f"Could not read {filename}: {e}")
            continue

        # Changing the filename extension to .csv:
        csv_filename = filename.rsplit(".", 1)[0] + ".csv"
        csv_path = os.path.join(output_folder, csv_filename)

        # Writing the dataframe to a csv file:
        try:
            df.to_csv(csv_path, index=False)
            print(f"Converted {filename} to {csv_filename}")
        except Exception as e:
            print(f"Could not convert {filename}: {e}")

print("Conversion complete.")

Now it's possible to save the filenames in a global variable to use them when needed with the right extension; the directory containing the files will be saved too:

In [ ]:
file_paths = [
    "../../DATA/CSV/2009-A riformattato.csv",
    "../../DATA/CSV/2010-A riformattato.csv",
    "../../DATA/CSV/2011-A riformattato.csv",
    "../../DATA/CSV/2012-A riformattato.csv",
    "../../DATA/CSV/2013-A riformattato.csv",
    "../../DATA/CSV/2014-A riformattato.csv",
    "../../DATA/CSV/2015-A riformattato.csv",
    "../../DATA/CSV/2016-A riformattato.csv",
    "../../DATA/CSV/2017-A riformattato.csv",
    "../../DATA/CSV/2018-A riformattato.csv",
    "../../DATA/CSV/2019-A riformattato.csv",
    "../../DATA/CSV/2020-A riformattato.csv",
    "../../DATA/CSV/2021-A riformattato.csv",
    "../../DATA/CSV/2022-A riformattato.csv",
    "../../DATA/CSV/2023-A riformattato.csv",
]
dir = "../../DATA/CSV" 

Now, it's important to uniform the names of the columns across files:

In [ ]:
import pandas as pd

# Defining uniform column names:
uniform_column_mapping = {
    "classe_donatore": "donor_class",
    "tipo": "donation_type",
    "nascita": "birth_year",
    "coorte_nascita": "birth_cohort",
    "prima_donazione": "first_donation_year",
    "coorte_prima_donazione": "first_donation_cohort",
    "numero_donazioni": "number_of_donations",
    "sesso": "gender",
    "data": "year",
    "ETÀ": "age",
    "ETA'": "age",
    "UN": "unique_number",
}

cleaned_files = []

for file_path in file_paths:
    # Loading the data:
    data = pd.read_csv(file_path)

    # Dropping the first column regardless of its name:
    data.drop(data.columns[0], axis=1, inplace=True)

    # Renaming columns to uniform names:
    data.rename(columns=uniform_column_mapping, inplace=True)

    # Standardizing data types - ensuring numeric columns are in the right data format:
    data["birth_year"] = data["birth_year"].astype(int, errors="ignore")
    data["birth_cohort"] = data["birth_cohort"].astype(int, errors="ignore")
    data["first_donation_year"] = data["first_donation_year"].astype(
        int, errors="ignore"
    )
    data["first_donation_cohort"] = data["first_donation_cohort"].astype(
        int, errors="ignore"
    )
    data["number_of_donations"] = data["number_of_donations"].astype(
        int, errors="ignore"
    )

    # Calculating missing 'age' data if possible:
    data["age"] = data.apply(
        lambda row: (
            row["year"] - row["birth_year"]
            if pd.isna(row["age"]) and row["birth_year"] > 0
            else row["age"]
        ),
        axis=1,
    )

    # Saving the cleaned data:
    data.to_csv(file_path, index=False)
    cleaned_files.append(file_path)

for file in cleaned_files:
    print(f"Cleaned and saved: {file}")

## Dataframe creation

Now that the files are all cleaned and formatted, it's possible to create a dataframe with them using **pandas**.

In [ ]:
# Creating a dataframe for each individual file:
dataframes = []

# Iterating over each file in the directory:
for file in file_paths:
    if file.endswith(".csv"):

        # Creating the dataframe for the current file:
        df = pd.read_csv(file)

        # Appending the dataframe to the list:
        dataframes.append(df)

# Creating the final dataframe by concatenating all the individual dataframes:
combined_dataframe = pd.concat(dataframes, ignore_index=True)

print("Displaying the first few rows of the combined DataFrame:")
print(combined_dataframe.head())

print("\nSummary Information about the Combined DataFrame:")
print(combined_dataframe.info())

print("\nDisplaying a random sample of 10 rows:")
print(combined_dataframe.sample(10))

### Dropping NaN values

Lines with missing values are usually dropped from the dataset; some values, however, can be calculated indirectly using other variables (e.g. "age"):

In [ ]:
print(combined_dataframe.info())
# Drop rows where any column has NaN values
combined_dataframe_cleaned = combined_dataframe.dropna()

# Display a summary to verify
print("Summary after dropping rows with missing values:")
print(combined_dataframe_cleaned.info())

Great. Now that the dataframe is ready to be used, it's possible to save it to a single `.csv` file for further analysis.

In [ ]:
combined_dataframe_cleaned.to_csv("../../DATA/FINAL/dataframe_cleaned.csv", index=False)